# L18 - Mini progetto finale

Lavoro a gruppi di 2 o 3 studenti. Tempo: 35 minuti.

Scegliere **una** delle due tracce e completare le sezioni del notebook nell'ordine. Ogni sezione indica che cosa fare; le celle con `None` vanno completate. La sezione 6 è la relazione, da scrivere in una cella di testo.

- **Traccia A - Punti nel piano.** Classificare punti di due classi disposti a spirale, con la rete scritta in NumPy della lezione L16 oppure con `MLPClassifier`.
- **Traccia B - Cifre scritte a mano.** Riconoscere le cifre 8 x 8 di scikit-learn con `MLPClassifier`.

Criteri di valutazione (dettaglio nella lezione): correttezza del codice, comprensione del modello, analisi dei risultati, comunicazione.

In [ ]:
# Funzione di controllo degli esercizi: eseguire questa cella, non modificarla.
def controlla(numero, prova):
    try:
        esito = bool(prova())
    except Exception as e:
        print(f"Esercizio {numero}: da rivedere ({type(e).__name__})")
        return
    print(f"Esercizio {numero}: " + ("corretto" if esito else "da rivedere"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

In [ ]:
TRACCIA = "B"

## 1. Dati

Eseguire la cella: carica i dati della traccia scelta in `X` (una riga per esempio) e `y` (etichette).

In [ ]:
def spirali(n=300, rumore=0.2, seme=0):
    """Due spirali intrecciate: n punti, etichette 0 e 1."""
    rng = np.random.default_rng(seme)
    m = n // 2
    t = np.sqrt(rng.uniform(0, 1, m)) * 3 * np.pi
    a = np.column_stack([t * np.cos(t), t * np.sin(t)]) / 3
    b = -a
    X = np.vstack([a, b]) + rng.normal(0, rumore, size=(2 * m, 2))
    y = np.concatenate([np.zeros(m), np.ones(m)]).astype(int)
    return X, y

if TRACCIA == "A":
    X, y = spirali()
    print("Traccia A:", X.shape, y.shape)
elif TRACCIA == "B":
    from sklearn.datasets import load_digits
    cifre = load_digits()
    X, y = cifre.data, cifre.target
    print("Traccia B:", X.shape, y.shape)
else:
    print("Assegnare \"A\" oppure \"B\" a TRACCIA e rieseguire le celle.")

## 2. Esplorazione dei dati

- Traccia A: disegnare i punti delle due classi con simboli diversi.
- Traccia B: mostrare almeno 10 immagini con la loro etichetta (`X[i].reshape(8, 8)`), e calcolare quante immagini ci sono per ogni cifra (`np.bincount(y)`).

Assegnare a `num_classi` il numero di classi presenti (`len(np.unique(y))`).

In [ ]:
num_classi = len(np.unique(y))
if TRACCIA == "A":
    plt.scatter(X[y == 0, 0], X[y == 0, 1], marker="o", s=12, label="classe 0")
    plt.scatter(X[y == 1, 0], X[y == 1, 1], marker="s", s=12, label="classe 1")
    plt.legend()
    plt.axis("equal")
    plt.show()
else:
    fig, assi = plt.subplots(1, 10, figsize=(12, 1.6))
    for i in range(10):
        assi[i].imshow(X[i].reshape(8, 8), cmap="gray_r")
        assi[i].set_title(str(y[i]))
        assi[i].axis("off")
    plt.show()
    print("immagini per cifra:", np.bincount(y))

In [ ]:
controlla("2 (esplorazione)", lambda: num_classi == len(np.unique(y)))

## 3. Preparazione

1. Normalizzare i dati: traccia A, standardizzazione per colonne (lezione L10); traccia B, dividere per 16 (i pixel valgono da 0 a 16).
2. Dividere in addestramento (70%) e verifica (30%) con `train_test_split(..., test_size=0.3, random_state=0)`.

Risultati: `X_add`, `X_ver`, `y_add`, `y_ver`.

In [ ]:
if TRACCIA == "A":
    Xn = (X - X.mean(axis=0)) / X.std(axis=0)
else:
    Xn = X / 16
X_add, X_ver, y_add, y_ver = train_test_split(Xn, y, test_size=0.3, random_state=0)
print(X_add.shape, X_ver.shape)

In [ ]:
controlla("3 (preparazione)", lambda: len(X_add) + len(X_ver) == len(X) and abs(len(X_ver) / len(X) - 0.3) < 0.01 and X_add.max() <= 10)

## 4. Modello e addestramento

Creare e addestrare il modello `modello` con `MLPClassifier` (traccia A: in alternativa si può usare la funzione `addestra_rete` della lezione L16, copiandola in una cella). Provare almeno **tre configurazioni diverse** (numero di neuroni o di strati nascosti, attivazione, tasso di apprendimento, numero di epoche) e annotare per ciascuna l'accuratezza sui dati di verifica nella tabella della relazione. Alla fine, lasciare in `modello` la configurazione migliore.

Disegnare la curva della perdita del modello scelto (`modello.loss_curve_`, disponibile con i solver `"sgd"` e `"adam"`).

In [ ]:
configurazioni = [(8,), (32,), (64, 32)]
for conf in configurazioni:
    m = MLPClassifier(hidden_layer_sizes=conf, max_iter=800, random_state=0)
    m.fit(X_add, y_add)
    print(conf, "addestramento:", round(m.score(X_add, y_add), 3), "verifica:", round(m.score(X_ver, y_ver), 3))

modello = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=800, random_state=0)
modello.fit(X_add, y_add)
plt.plot(modello.loss_curve_)
plt.xlabel("epoca")
plt.ylabel("perdita")
plt.title("Curva della perdita")
plt.show()

## 5. Valutazione

Calcolare `acc_add` e `acc_ver`, le accuratezze del modello scelto sui dati di addestramento e di verifica.

- Traccia A: disegnare la regione di decisione (come nella lezione L16, con `np.meshgrid` e `modello.predict_proba`).
- Traccia B: mostrare alcune immagini classificate male, con la cifra vera e quella prevista.

In [ ]:
acc_add = modello.score(X_add, y_add)
acc_ver = modello.score(X_ver, y_ver)
print("addestramento:", acc_add, " verifica:", acc_ver)
if TRACCIA == "B":
    prev = modello.predict(X_ver)
    errati = np.where(prev != y_ver)[0]
    fig, assi = plt.subplots(1, min(6, len(errati)), figsize=(9, 1.8))
    for a, i in zip(np.atleast_1d(assi), errati[:6]):
        a.imshow(X_ver[i].reshape(8, 8), cmap="gray_r")
        a.set_title(f"vera {y_ver[i]}, prev. {prev[i]}", fontsize=8)
        a.axis("off")
    plt.show()
else:
    g1 = np.linspace(X_ver[:, 0].min() - 0.5, X_ver[:, 0].max() + 0.5, 200)
    g2 = np.linspace(X_ver[:, 1].min() - 0.5, X_ver[:, 1].max() + 0.5, 200)
    G1, G2 = np.meshgrid(g1, g2)
    prob = modello.predict_proba(np.column_stack([G1.ravel(), G2.ravel()]))[:, 1].reshape(G1.shape)
    plt.contourf(G1, G2, prob, levels=20, cmap="Greys", alpha=0.6)
    plt.scatter(X_add[y_add == 0, 0], X_add[y_add == 0, 1], marker="o", s=12, label="classe 0")
    plt.scatter(X_add[y_add == 1, 0], X_add[y_add == 1, 1], marker="s", s=12, label="classe 1")
    plt.legend()
    plt.title("Regione di decisione")
    plt.show()

In [ ]:
controlla("5 (valutazione)", lambda: np.isclose(acc_ver, modello.score(X_ver, y_ver)) and acc_ver > (0.85 if TRACCIA == "A" else 0.9))

## 6. Relazione

Completare la cella seguente (doppio clic per modificarla).

**Gruppo:** (nomi)

**Traccia scelta:** A / B

**Configurazioni provate:**

| Configurazione | Accuratezza addestramento | Accuratezza verifica |
|---|---|---|
| | | |
| | | |
| | | |

**Modello scelto e motivazione** (2-3 righe):

**Andamento della perdita** (che cosa mostra la curva?):

**Differenza tra accuratezza di addestramento e di verifica** (è grande o piccola? che cosa significa?):

**Errori del modello** (quali esempi sbaglia? perché potrebbe sbagliarli?):

**Che cosa si potrebbe provare per migliorare** (almeno un'idea):